In [ ]:
!pip install -q openpyxl

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from IPython.display import display

In [ ]:
file_path = "/content/Cleaned_Cutoff_Data.xlsx"  # Upload this file before running
df = pd.read_excel(file_path)

In [ ]:
le_course = LabelEncoder()
le_category = LabelEncoder()

df['Course'] = le_course.fit_transform(df['Course Name'])
df['Category_Encoded'] = le_category.fit_transform(df['Category'])

In [ ]:

def parse_input(text):
    percentage_match = re.search(r"(\d{2}\.?\d*)\s?%", text)
    percentage = float(percentage_match.group(1)) if percentage_match else None

    course_keywords = df['Course Name'].unique().tolist()
    category_keywords = df['Category'].unique().tolist()

    course = next((c for c in course_keywords if c.lower() in text.lower()), None)
    category = next((c for c in category_keywords if c.lower() in text.lower()), None)

    return percentage, category, course


In [ ]:

def create_training_data(df, user_percent):
    df = df.copy()
    df['Selected'] = df['Percentage'].apply(lambda x: 1 if user_percent >= x else 0)

    # Ensure we have both classes
    if df['Selected'].nunique() == 1:
        df.loc[df.sample(frac=0.5).index, 'Selected'] = 1 - df['Selected'].iloc[0]

    return df


In [ ]:

def train_model(df):
    X = df[['Course', 'Category_Encoded', 'Percentage']]
    y = df['Selected']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    return model


In [ ]:

def recommend(model, user_percent, category, course, le_course, le_category, df):
    df = df.copy()
    df['Course'] = le_course.transform(df['Course Name'])
    df['Category_Encoded'] = le_category.transform(df['Category'])

    X_user = df[['Course', 'Category_Encoded', 'Percentage']]
    proba = model.predict_proba(X_user)

    if proba.shape[1] == 2:
        df['Probability'] = proba[:, 1]
    else:
        fallback_val = 1.0 if model.classes_[0] == 1 else 0.0
        df['Probability'] = fallback_val

    top_10 = df.sort_values(by='Probability', ascending=False).head(10)
    bottom_10 = df.sort_values(by='Probability', ascending=True).head(10)

    return top_10[['College Name', 'Course Name', 'Category', 'Percentage', 'Probability']],            bottom_10[['College Name', 'Course Name', 'Category', 'Percentage', 'Probability']]


In [ ]:
# Sample user input
user_input = "Suggest top 10 Electrical Engineering colleges for 88% in CAP category"

# Parse input
percent, category, course = parse_input(user_input)

if percent and course and category:
    print(f"🔍 Parsed input:\nCourse: {course}\nCategory: {category}\nPercentage: {percent}")

    df_encoded = df.copy()
    df_encoded['Course'] = le_course.transform(df_encoded['Course Name'])
    df_encoded['Category_Encoded'] = le_category.transform(df_encoded['Category'])

    df_prepared = create_training_data(df_encoded, percent)
    model = train_model(df_prepared)

    top, bottom = recommend(model, percent, category, course, le_course, le_category, df)

    print("\n🎯 Top 10 Colleges You May Get:")
    display(top)

    print("\n❌ Colleges You Might Not Get:")
    display(bottom)
else:
    print("❌ Could not extract course, category, or percentage from input.")

🔍 Parsed input:
Course: Electrical Engineering
Category: CAP
Percentage: 88.0

🎯 Top 10 Colleges You May Get:


,College Name,Course Name,Category,Percentage,Probability
1,"1002 Government College of Engineering, Amrava...",Civil Engineering,EWS,85.13,1.0
3,"1002 Government College of Engineering, Amrava...",Electrical Engineering,CAP,87.16,1.0
8,"1005 Sant Gadge Baba Amravati University,Amrav...",Petro Chemical Engineering,VII,82.11,1.0
7,"1002 Government College of Engineering, Amrava...",Mechanical Engineering,LSEBC,52.12,1.0
6,"1002 Government College of Engineering, Amrava...",Electronics and Telecommunication Engg,VII,83.00,1.0
11,"1012 Government College of Engineering,Yavatma...",Civil Engineering,AY,79.58,1.0
10,"1012 Government College of Engineering,Yavatma...",Civil Engineering,II,75.32,1.0
9,"1012 Government College of Engineering,Yavatma...",Civil Engineering,CAP,85.58,1.0
17,"1012 Government College of Engineering,Yavatma...",Mechanical Engineering,CAP,77.79,1.0
16,"1012 Government College of Engineering,Yavatma...",Electronics and Telecommunication Engg,VII,84.71,1.0



❌ Colleges You Might Not Get:


,College Name,Course Name,Category,Percentage,Probability
20,1101 Shri Sant Gajanan Maharaj College of Engi...,Information Technology,LOBC,88.74,0.37
12,"1012 Government College of Engineering,Yavatma...",Computer Engineering,VII,89.56,0.71
4,"1002 Government College of Engineering, Amrava...",Electrical Engineering,II,88.79,0.76
0,"1002 Government College of Engineering, Amrava...",Civil Engineering,GSEBC,88.37,0.76
2,"1002 Government College of Engineering, Amrava...",Information Technology,LSC,86.13,0.95
5,"1002 Government College of Engineering, Amrava...",Electrical Engineering,AY,87.67,0.98
32,1105 Prof. Ram Meghe Institute of Technology &...,Information Technology,VII,85.26,0.98
67,1114 Sipna Shikshan Prasarak Mandal College of...,Information Technology,II,74.31,0.99
88,1119 Paramhansa Ramkrishna Maunibaba Shikshan ...,Information Technology,II,71.26,0.99
8,"1005 Sant Gadge Baba Amravati University,Amrav...",Petro Chemical Engineering,VII,82.11,1.00
